<a href="https://colab.research.google.com/github/Adyypower/Deep-learning-Models-or-topics/blob/main/BERT_encoder_model(questioning_answering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from transformers import (
    BertForQuestionAnswering,
    BertTokenizerFast,
)

In [2]:
from scipy.special import softmax
import plotly.express as px
import pandas as pd
import numpy as np

In [3]:
context = "The giraffe is a large African hoofed mammal belonging to the genus Giraffa. It is the tallest living terrestrial animal and the largest ruminant on Earth. It is classified under the family Giraffidae, along with its closest extant relative, the okapi. Traditionally, giraffes have been thought of as one species, Giraffa camelopardalis, with nine subspecies. Most recently, researchers proposed dividing them into four extant species, with seven subspecies, which can be distinguished morphologically by their fur coat patterns. Six valid extinct species of Giraffa are known from the fossil record."

In [4]:
question = " how many giraffe species are there?"

In [5]:
model = "deepset/bert-base-cased-squad2"
tokenizer = BertTokenizerFast.from_pretrained(model)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [6]:
model = BertForQuestionAnswering.from_pretrained(model)

Some weights of the model checkpoint at deepset/bert-base-cased-squad2 were not used when initializing BertForQuestionAnswering: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForQuestionAnswering from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForQuestionAnswering from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [7]:
inputs = tokenizer(question,context,return_tensors="pt")

In [10]:
with torch.no_grad():
    outputs = model(**inputs)

In [11]:
outputs

QuestionAnsweringModelOutput(loss=None, start_logits=tensor([[-2.0263, -7.8823, -8.4305, -7.7608, -8.5623, -8.9115, -9.0814, -8.4639,
         -8.3083, -8.3920, -8.8604, -8.2153, -4.4070, -8.2658, -6.6317, -8.5478,
         -7.1309, -5.9283, -7.7713, -6.6344, -8.4573, -9.1667, -7.1283, -7.4830,
         -8.4075, -8.7134, -8.4266, -8.1987, -7.0159, -8.6464, -7.9972, -8.9776,
         -8.5201, -6.3675, -8.2389, -7.6629, -6.2666, -8.0571, -7.4042, -8.7212,
         -8.7643, -7.8957, -7.2784, -7.4212, -8.6217, -9.4804, -8.9356, -8.0112,
         -8.8358, -6.1175, -8.2808, -8.0013, -8.3315, -8.2221, -7.2150, -6.5524,
         -8.7002, -8.2108, -8.7155, -9.6038, -9.0035, -9.1916, -8.0862, -7.8632,
         -8.5981, -8.5045, -9.1827, -8.0576, -6.6299, -8.4610, -9.0904, -8.1933,
         -7.7022, -8.1257, -4.0176, -7.9397, -7.1914, -8.1065, -7.8103, -8.2774,
         -7.3492, -8.7059, -7.8376, -2.3020, -6.4270, -9.2126, -5.3236, -8.2177,
         -7.3417, -8.6444, -6.3205, -8.6159, -8.5776, -9

In [9]:
import torch

In [12]:
start_scores, end_scores = softmax(outputs.start_logits)[0], softmax(outputs.end_logits)[0]

In [13]:
score_df = pd.DataFrame({
    'token position': list(range(len(start_scores)))*2,
    'score': list(start_scores)+list(end_scores),
    'score Type': ['start']*len(start_scores)+['end']*len(end_scores)
})
px.bar(score_df,x ='token position',y = 'score',color = 'score Type',barmode = 'group')

In [14]:
start_index = np.argmax(start_scores)
end_index = np.argmax(end_scores)



In [15]:
answer_ids = inputs.input_ids[0][start_index:end_index+1]
answer_tokens = tokenizer.convert_ids_to_tokens(answer_ids)
answer = tokenizer.convert_tokens_to_string(answer_tokens)
answer

'four'

In [16]:
# Part 2
# Defining a function to predict the answer to a question given a context
def predict_answer(context, question):
    inputs = tokenizer(question, context, return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        outputs = model(**inputs)
    start_scores, end_scores = softmax(outputs.start_logits)[0], softmax(outputs.end_logits)[0]
    start_idx = np.argmax(start_scores)
    end_idx = np.argmax(end_scores)
    confidence_score = (start_scores[start_idx] + end_scores[end_idx]) /2
    answer_ids = inputs.input_ids[0][start_idx: end_idx + 1]
    answer_tokens = tokenizer.convert_ids_to_tokens(answer_ids)
    answer = tokenizer.convert_tokens_to_string(answer_tokens)
    if answer != tokenizer.cls_token:
        return answer, confidence_score
    return None, confidence_score


In [ ]:
output = predict_answer(question,context)
output

In [17]:
# Defining a new context and predicting answers to some questions
context = """Coffee is a beverage brewed from roasted coffee beans. Darkly colored, bitter, and slightly acidic, coffee has a stimulating effect on humans, primarily due to its caffeine content. It has the highest sales in the world market for hot drinks.[2]

The seeds of the Coffea plant's fruits are separated to produce unroasted green coffee beans. The beans are roasted and then ground into fine particles typically steeped in hot water before being filtered out, producing a cup of coffee. It is usually served hot, although chilled or iced coffee is common. Coffee can be prepared and presented in a variety of ways (e.g., espresso, French press, caffè latte, or already-brewed canned coffee). Sugar, sugar substitutes, milk, and cream are often added to mask the bitter taste or enhance the flavor.

Though coffee is now a global commodity, it has a long history tied closely to food traditions around the Red Sea. The earliest credible evidence of coffee drinking as the modern beverage appears in modern-day Yemen in southern Arabia in the middle of the 15th century in Sufi shrines, where coffee seeds were first roasted and brewed in a manner similar to how it is now prepared for drinking.[3] The coffee beans were procured by the Yemenis from the Ethiopian Highlands via coastal Somali intermediaries, and cultivated in Yemen. By the 16th century, the drink had reached the rest of the Middle East and North Africa, later spreading to Europe.

The two most commonly grown coffee bean types are C. arabica and C. robusta.[4] Coffee plants are cultivated in over 70 countries, primarily in the equatorial regions of the Americas, Southeast Asia, the Indian subcontinent, and Africa. As of 2023, Brazil was the leading grower of coffee beans, producing 35% of the world's total. Green, unroasted coffee is traded as an agricultural commodity. Despite sales of coffee reaching billions of dollars worldwide, farmers producing coffee beans disproportionately live in poverty. Critics of the coffee industry have also pointed to its negative impact on the environment and the clearing of land for coffee-growing and water use. The global coffee industry is massive and worth $495.50 billion as of 2023.[5] Brazil, Vietnam, and Colombia are the top exporters of coffee beans as of 2023. \n Rapid growth in coffee production in South America during the second half of the 19th century was matched by an increase in consumption in developed countries, though nowhere has this growth been as pronounced as in the United States, where a high rate of population growth was compounded by doubling of per capita consumption between 1860 and 1920. Though the United States was not the heaviest coffee-drinking nation at the time (Belgium, the Netherlands and Nordic countries all had comparable or higher levels of per capita consumption), due to its sheer size, it was already the largest consumer of coffee in the world by 1860, and, by 1920, around half of all coffee produced worldwide was consumed in the US.[37]

Coffee has become a vital cash crop for many developing countries. Over one hundred million people in developing countries have become dependent on coffee as their primary source of income. It has become the primary export and economic backbone for African countries like Uganda, Burundi, Rwanda, and Ethiopia,[40] as well as many Central American countries.

"""

len(tokenizer.tokenize(context))
predict_answer(context, "What is coffee?")
predict_answer(context, "What are the most common coffee beans?")
predict_answer(context, "How can I make ice coffee?")
predict_answer(context[4000:], "How many people are dependent on coffee for their income?")


Token indices sequence length is longer than the specified maximum sequence length for this model (714 > 512). Running this sequence through the model will result in indexing errors


(None, np.float32(0.99884295))

In [18]:
# Defining a function to chunk sentences
def chunk_sentences(sentences, chunk_size, stride):
    chunks = []
    num_sentences = len(sentences)
    for i in range(0, num_sentences, chunk_size - stride):
        chunk = sentences[i: i + chunk_size]
        chunks.append(chunk)
    return chunks




In [19]:
sentences = [
    "Sentence 1.",
    "Sentence 2.",
    "Sentence 3.",
    "Sentence 4.",
    "Sentence 5.",
    "Sentence 6.",
    "Sentence 7.",
    "Sentence 8.",
    "Sentence 9.",
    "Sentence 10."
]


sentences = context.split("\n")
chunked_sentences = chunk_sentences(sentences, chunk_size=3, stride=1)


questions = ["What is coffee?", "What are the most common coffee beans?", "How can I make ice coffee?", "How many people are dependent on coffee for their income?"]

answers = {}

for chunk in chunked_sentences:
    sub_context = "\n".join(chunk)
    for question in questions:
        answer, score = predict_answer(sub_context, question)

        if answer:
            if question not in answers:
                answers[question] = (answer, score)
            else:
                if score > answers[question][1]:
                    answers[question] = (answer, score)

print(answers)

{'What is coffee?': ('a beverage brewed from roasted coffee beans', np.float32(0.9282471)), 'What are the most common coffee beans?': ('C. arabica and C. robusta', np.float32(0.948103)), 'How many people are dependent on coffee for their income?': ('Over one hundred million', np.float32(0.8877787))}
